# Generate Embeddings - Learning Exercise

This notebook demonstrates how vector embeddings are generated using Azure OpenAI's API.

## Learning Objectives
- Understand the structure of Airbnb listing data
- Generate vector embeddings using Azure OpenAI's `text-embedding-3-small` deployment
- See how text is converted to 1536-dimension vectors

**⏭️ Note:** This is a **learning demonstration** that does not modify the application or database. The full dataset is already pre-embedded in `data/embedded_data.json`. If you're short on time, you can skip directly to [vector-search.ipynb](./vector-search.ipynb).

## Step 1: Understanding the Data

Our dataset contains Airbnb listings with the following key fields:

| Field | Type | Description | Example |
|-------|------|-------------|----------|
| `id` | number | Unique identifier | `360` |
| `listing_url` | string | URL to the listing | `"https://www.airbnb.com/rooms/360"` |
| `name` | string | Property title | `"Chickadee Cottage in LoHi"` |
| `description` | string | Full description | Text used for embeddings |
| `neighborhood_overview` | string | Area information | `"Located in Lower Highlands..."` |
| `amenities` | array | List of amenities | `["Wifi", "Kitchen", "TV", ...]` |
| `property_type` | string | Type of property | `"Entire guesthouse"`, `"Apartment"`, etc. |
| `bedrooms` | number | Number of bedrooms | `1`, `2`, `3`, etc. |
| `price` | number | Nightly price | `161.0` |
| `latitude` | number | Latitude coordinate | `39.766414` |
| `longitude` | number | Longitude coordinate | `-105.002098` |

### Composite Embedding Strategy

Rather than embedding just the `description` field, we build a **composite text** from multiple fields for richer semantic signal:

```
<name>. <property_type>. <cleaned description>
Neighborhood: <neighborhood_overview>
Amenities: <top amenities>
```

This approach:
- Captures property identity (`name`, `property_type`)
- Strips noise (license numbers, fee policies, markdown formatting)
- Adds spatial context (`neighborhood_overview`)
- Includes search-relevant amenities

### Pre-embedded Data

For this workshop, we provide **pre-embedded data** in `data/embedded_data.json` that already contains the `descriptionVector` field. This saves time and API costs.

However, understanding how embeddings are generated is essential! In the next steps, we'll demonstrate embedding 50 sample documents as a learning exercise.

In [ ]:
import os
import json
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# Initialize Azure OpenAI client
openai_client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT')
)

print("✅ Libraries imported and Azure OpenAI environment loaded")

## Step 2: Load and Examine Sample Data

Let's explore the raw dataset structure to understand what fields are available before embeddings are added.

In [ ]:
# Load raw data (without embeddings)
with open('../data/raw_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"📊 Loaded {len(data)} listings from raw_data.json")

# Examine the first listing
sample = data[0]
print(f"\n📄 Sample Listing:")
print(f"   ID: {sample['id']}")
print(f"   Name: {sample['name']}")
print(f"   Property Type: {sample['property_type']}")
print(f"   Bedrooms: {sample.get('bedrooms', 'N/A')}")
print(f"   Price: ${sample.get('price', 'N/A')}")
print(f"   Amenities: {', '.join(sample.get('amenities', [])[:5])}...")
print(f"\n📝 Description Preview:")
print(f"   {sample.get('description', '')[:200]}...")

## Step 3: Create Embedding Generation Function

We'll use Azure OpenAI's `text-embedding-3-small` deployment to generate 1536-dimension vectors that capture semantic meaning.

### 💡 Understanding the Embedding

Each number in the 1536-dimension vector represents a learned feature. The model has discovered that certain combinations of these numbers correspond to semantic concepts like "cozy", "parking", "downtown", etc.

In [ ]:
def generate_embedding(text):
    """
    Generate a vector embedding for the given text using Azure OpenAI.
    
    Args:
        text (str): The text to embed
        
    Returns:
        list: A 1536-dimension vector representing the text
    """
    if not text or not isinstance(text, str):
        return None
    
    try:
        response = openai_client.embeddings.create(
            model=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-small'),
            input=text
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Error generating embedding: {e}")
        return None


# Test the function with a sample query
test_text = "Cozy apartment near downtown with free parking"
test_embedding = generate_embedding(test_text)

print(f"🧪 Testing Embedding Generation:")
print(f"   Input: '{test_text}'")
print(f"   ✅ Generated embedding")
print(f"   📏 Dimensions: {len(test_embedding)}")
print(f"   📊 First 5 values: {test_embedding[:5]}")

## Step 4: Generate Embeddings for 50 Documents

Now let's embed 50 documents from `raw_data.json` to understand the full process.

**Note:** This is a learning exercise - we won't write to the database since pre-embedded data (`data/embedded_data.json`) is already available.

In [ ]:
def embed_documents(documents, limit=50):
    """
    Generate embeddings for a list of documents.
    
    Args:
        documents (list): List of listing documents
        limit (int): Maximum number of documents to process
        
    Returns:
        list: Documents with descriptionVector added
    """
    docs_to_process = documents[:limit]
    embedded_docs = []
    
    print(f"🔄 Generating embeddings for {len(docs_to_process)} documents...")
    
    for idx, doc in enumerate(docs_to_process):
        description = doc.get('description', '')
        embedding = generate_embedding(description)
        
        if embedding:
            doc_copy = doc.copy()
            doc_copy['descriptionVector'] = embedding
            embedded_docs.append(doc_copy)
        
        # Progress update every 10 documents
        if (idx + 1) % 10 == 0:
            print(f"   ✅ Processed {idx + 1}/{len(docs_to_process)} documents...")
    
    print(f"\n✅ Generated embeddings for {len(embedded_docs)} documents")
    return embedded_docs

## Step 5: Run Embedding Process and Verify Results

Run the embedding function on 50 documents to see the process in action.

In [ ]:
# Run the embedding process on 50 documents
embedded_documents = embed_documents(data, limit=50)

# Show results summary
print(f"\n📊 Results Summary:")
print(f"   Documents processed: {len(embedded_documents)}")
print(f"   Embedding dimensions: {len(embedded_documents[0]['descriptionVector'])}")

In [ ]:
# Show a sample embedded document
sample_embedded = embedded_documents[0]
print(f"📄 Sample Embedded Document:")
print(f"   Name: {sample_embedded['name']}")
print(f"   Has embedding: {'descriptionVector' in sample_embedded}")
print(f"   Vector preview: {sample_embedded['descriptionVector'][:3]}...")

print("\n" + "=" * 60)
print("✨ Learning exercise complete!")
print("   The full dataset is pre-embedded in data/embedded_data.json")
print("=" * 60)

## 🎉 Next Steps

Now that you understand how embeddings are generated, continue to **[Module 1](../exercises/Module-01.md#-step-6-create-vector-index-using-mongodb-atlas)** to: